# Data Visualization 101
## Diamonds Dataset Guide for Matplotlib, Seaborn, and Plotly

Dataset used throughout: `seaborn.load_dataset("diamonds")`

## Topic flow

1. Data load and quick inspection  
2. Matplotlib visuals  
3. Seaborn visuals  
4. Plotly visuals  
5. Revision questions

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
px.defaults.template = "plotly_white"

df = sns.load_dataset("diamonds").copy()

cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

df["cut"] = pd.Categorical(df["cut"], categories=cut_order, ordered=True)
df["color"] = pd.Categorical(df["color"], categories=color_order, ordered=True)
df["clarity"] = pd.Categorical(df["clarity"], categories=clarity_order, ordered=True)

sample_df = df.sample(5000, random_state=42)
swarm_df = df.sample(500, random_state=42)
pair_df = df.sample(1500, random_state=42)
matrix_df = df.sample(2000, random_state=42)

cut_summary = (
    df.groupby("cut", observed=True)
      .agg(
          avg_price=("price", "mean"),
          avg_carat=("carat", "mean"),
          avg_depth=("depth", "mean"),
          avg_table=("table", "mean"),
          count=("price", "size"),
      )
      .reindex(cut_order)
      .reset_index()
)

color_summary = (
    df.groupby("color", observed=True)
      .agg(
          avg_price=("price", "mean"),
          avg_carat=("carat", "mean"),
          avg_depth=("depth", "mean"),
          avg_table=("table", "mean"),
          count=("price", "size"),
      )
      .reindex(color_order)
      .reset_index()
)

clarity_summary = (
    df.groupby("clarity", observed=True)
      .agg(
          avg_price=("price", "mean"),
          avg_carat=("carat", "mean"),
          avg_depth=("depth", "mean"),
          avg_table=("table", "mean"),
          count=("price", "size"),
      )
      .reindex(clarity_order)
      .reset_index()
)

In [ ]:
print(df.shape)
display(df.head())
display(df.describe().T)
display(df.describe(include="category").T)
display(df.isna().sum())

## Quick revision questions

1. Which columns are numerical?  
2. Which columns are categorical?  
3. Which variables look ordinal?  
4. Which column will work well as a target for price-based comparisons?

# 1. Matplotlib visuals

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(color_summary["color"], color_summary["avg_price"], marker="o")
ax.set_title("Average Price by Color")
ax.set_xlabel("Color")
ax.set_ylabel("Average Price")
plt.show()

### Question
How does average price move across the color grades?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["carat"], df["price"], alpha=0.3, s=12)
ax.set_title("Carat vs Price")
ax.set_xlabel("Carat")
ax.set_ylabel("Price")
plt.show()

### Question
Does price increase as carat increases?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cut_summary["cut"].astype(str), cut_summary["avg_price"])
ax.set_title("Average Price by Cut")
ax.set_xlabel("Cut")
ax.set_ylabel("Average Price")
plt.show()

### Question
Which cut has the highest average price?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["price"], bins=50)
ax.set_title("Price Distribution")
ax.set_xlabel("Price")
ax.set_ylabel("Frequency")
plt.show()

### Question
What does the price distribution look like?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([df.loc[df["cut"] == c, "price"] for c in cut_order], tick_labels=cut_order)
ax.set_title("Price by Cut")
ax.set_xlabel("Cut")
ax.set_ylabel("Price")
plt.show()

### Question
Which cut has the widest spread and the most outliers?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(color_summary["color"], color_summary["avg_price"])
ax.set_title("Area View of Average Price by Color")
ax.set_xlabel("Color")
ax.set_ylabel("Average Price")
plt.show()

### Question
How do average prices move across the color sequence?

In [ ]:
cut_counts = df["cut"].value_counts().reindex(cut_order)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(cut_counts, labels=cut_counts.index.astype(str), autopct="%1.1f%%")
ax.set_title("Cut Share")
plt.show()

### Question
Which cut occupies the largest share?

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, polar=True)

angles = np.linspace(0, 2 * np.pi, len(color_summary), endpoint=False).tolist()
values = color_summary["avg_price"].tolist()

angles += angles[:1]
values += values[:1]

ax.plot(angles, values, marker="o")
ax.fill(angles, values, alpha=0.25)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(color_summary["color"].astype(str))
ax.set_title("Polar View of Average Price by Color")
plt.show()

### Question
Is a polar chart easy to read for this comparison?

In [ ]:
radar_cols = ["avg_price", "avg_carat", "avg_depth", "avg_table"]
radar_data = cut_summary[["cut"] + radar_cols].copy()

norm = radar_data[radar_cols].copy()
norm = (norm - norm.min()) / (norm.max() - norm.min())
norm["cut"] = radar_data["cut"]

label_values = radar_cols
angles = np.linspace(0, 2 * np.pi, len(label_values), endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, polar=True)

for _, row in norm.iterrows():
    vals = row[label_values].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=str(row["cut"]))
    ax.fill(angles, vals, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(["Avg Price", "Avg Carat", "Avg Depth", "Avg Table"])
ax.set_title("Radar Chart of Cut Groups (Normalized)")
ax.legend(bbox_to_anchor=(1.15, 1.1))
plt.show()

### Question
How do cut groups compare across price, carat, depth, and table?

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    sample_df["carat"],
    sample_df["depth"],
    sample_df["price"],
    c=sample_df["price"],
    cmap="viridis",
    alpha=0.35,
    s=10
)

ax.set_xlabel("Carat")
ax.set_ylabel("Depth")
ax.set_zlabel("Price")
ax.set_title("3D Scatter: Carat, Depth, Price")
plt.show()

### Practice questions — Matplotlib

1. Which chart is best for a single numerical variable?  
2. Which chart is best for comparing categories on price?  
3. Which chart is best for outliers?  
4. Which chart is best for a simple 3D view?

# 2. Seaborn visuals

In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Count")
plt.show()

### Question
How is price distributed?

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x="cut", order=cut_order)
plt.title("Count of Cut")
plt.xlabel("Cut")
plt.ylabel("Count")
plt.show()

### Question
Which cut appears most often?

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample_df, x="carat", y="price", hue="cut", alpha=0.5, s=25)
plt.title("Carat vs Price")
plt.xlabel("Carat")
plt.ylabel("Price")
plt.legend(title="Cut")
plt.show()

### Question
Does cut change the carat-price pattern?

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=cut_summary, x="cut", y="avg_price", order=cut_order)
plt.title("Average Price by Cut")
plt.xlabel("Cut")
plt.ylabel("Average Price")
plt.show()

### Question
Which cut has the highest average price?

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="cut", y="price", order=cut_order)
plt.title("Price by Cut")
plt.xlabel("Cut")
plt.ylabel("Price")
plt.show()

### Question
Which cut has the widest spread and the highest outliers?

In [ ]:
plt.figure(figsize=(8, 4))
sns.violinplot(data=df, x="cut", y="price", order=cut_order, inner="box")
plt.title("Price Distribution by Cut")
plt.xlabel("Cut")
plt.ylabel("Price")
plt.show()

### Question
How is the full distribution of price changing across cut?

In [ ]:
plt.figure(figsize=(8, 4))
sns.swarmplot(data=swarm_df, x="cut", y="carat", order=cut_order, size=3)
plt.title("Carat by Cut")
plt.xlabel("Cut")
plt.ylabel("Carat")
plt.show()

### Question
Where do individual carat values sit across the cut groups?

In [ ]:
corr = df[["carat", "depth", "table", "price", "x", "y", "z"]].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

### Question
Which numerical variables move together?

In [ ]:
sns.jointplot(data=sample_df, x="carat", y="price", kind="hex", height=6)
plt.show()

### Question
How do carat and price behave together?

In [ ]:
sns.pairplot(
    pair_df,
    vars=["carat", "depth", "table", "price"],
    hue="cut",
    diag_kind="hist"
)
plt.show()

### Question
Which numerical variables are related?

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    sample_df["carat"],
    sample_df["depth"],
    sample_df["price"],
    c=sample_df["price"],
    cmap="viridis",
    alpha=0.35,
    s=10
)

ax.set_xlabel("Carat")
ax.set_ylabel("Depth")
ax.set_zlabel("Price")
ax.set_title("3D Scatter with Seaborn Theme")
plt.show()

### Question
How do carat, depth, and price look together in 3D?

### Practice questions — Seaborn

1. Which plot is best for comparing distributions across categories?  
2. Which plot is best for checking correlation?  
3. Which plot is best for point-level category comparison?  
4. Which plot is best for a two-variable distribution view?

# 3. Plotly visuals

In [ ]:
px.defaults.template = "plotly_white"

In [ ]:
fig = px.histogram(df, x="price", nbins=50, title="Interactive Price Distribution")
fig.show()

### Question
How does interactivity change the distribution view?

In [ ]:
fig = px.scatter(
    sample_df,
    x="carat",
    y="price",
    color="cut",
    size="depth",
    hover_data=["color", "clarity"],
    title="Interactive Carat vs Price"
)
fig.show()

### Question
Does color help separate the groups?

In [ ]:
fig = px.box(df, x="cut", y="price", color="cut", title="Price by Cut")
fig.show()

### Question
Which cut shows the highest median price?

In [ ]:
fig = px.violin(df, x="cut", y="price", color="cut", box=True, points="all", title="Price Distribution by Cut")
fig.show()

### Question
How does the full distribution compare across cut groups?

In [ ]:
fig = px.bar(cut_summary, x="cut", y="avg_price", text="avg_price", title="Average Price by Cut")
fig.show()

### Question
Which cut has the highest average price after aggregation?

In [ ]:
fig = px.density_heatmap(
    sample_df,
    x="carat",
    y="price",
    nbinsx=40,
    nbinsy=40,
    title="Density of Carat and Price"
)
fig.show()

### Question
Where are the dense regions in the carat-price map?

In [ ]:
fig = px.treemap(
    df,
    path=["cut", "color", "clarity"],
    title="Treemap: Cut → Color → Clarity"
)
fig.show()

### Question
How do the categories form a hierarchy?

In [ ]:
fig = px.scatter_matrix(
    matrix_df,
    dimensions=["carat", "depth", "table", "price", "x", "y", "z"],
    color="cut",
    title="Scatter Matrix"
)
fig.show()

### Question
Which pair of numerical variables looks strongest?

In [ ]:
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Interactive Correlation Heatmap"
)
fig.show()

### Question
Which variables are closely related to price?

In [ ]:
fig = px.scatter_3d(
    sample_df,
    x="carat",
    y="depth",
    z="price",
    color="cut",
    size="table",
    hover_data=["color", "clarity"],
    title="3D Scatter Plot"
)
fig.show()

### Question
How do three numerical variables behave together?

In [ ]:
fig = px.parallel_coordinates(
    sample_df,
    dimensions=["carat", "depth", "table", "price", "x", "y", "z"],
    color=sample_df["price"],
    title="Parallel Coordinates"
)
fig.show()

### Question
How do many numerical variables vary together?

In [ ]:
fig = px.parallel_categories(
    sample_df,
    dimensions=["cut", "color", "clarity"],
    color=sample_df["price"],
    title="Parallel Categories"
)
fig.show()

### Question
How do the categorical groups connect?

### Practice questions — Plotly

1. Which plot is best for interactive exploration of price?  
2. Which plot is best for hierarchy?  
3. Which plot is best for multivariate numerical comparison?  
4. Which plot is best for a 3D view with hover?

# 4. Revision sheet

## One numerical variable
Histogram, box plot, violin plot, KDE

## One categorical variable
Count plot, bar chart, pie chart, treemap

## Two numerical variables
Scatter plot, regression line, joint plot, density heatmap

## Numerical + categorical
Box plot, violin plot, swarm plot, grouped bar chart

## Two categorical variables
Crosstab heatmap, treemap, sunburst

## Many variables
Pair plot, scatter matrix, heatmap, parallel coordinates, 3D scatter